> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 01. ROS 2 통신 기본 — Node·Topic·Message / 60분

**선행:** Python 함수와 dict, 08 README의 결정적/실제 ROS 경로 구분

**이 노트북의 결과:** sensor node가 이름 있는 topic으로 고정 schema message를 보내고 monitor callback이 받는 과정을 설명합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

ROS는 하나의 큰 프로그램을 여러 node로 나누고 이름과 타입이 정해진 메시지로 연결하는 방식입니다. 먼저 Python 모형에서 메시지 계약을 이해하고, 실제 ROS에서 빌드·환경 설정·발행·수신의 경계를 확인합니다.

### 전체 과정에서 내 코드가 하는 일

```text
메시지 TODO·모형 검증
  ↓
package build
  ↓
각 Bash 환경 설정
  ↓
A 발행
  ↓
B 수신
  ↓
C 토픽 관찰
```

필드·타입·값을 검사하는 함수를 구현합니다. 실제 ROS는 별도 터미널에서 실행하며 세 터미널의 Domain을 맞춥니다. 모형 PASS가 DDS 통신 성공을 대신하지 않습니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** package 파일이 있는데 ros2가 package를 못 찾으면?

**설명:** 소스 존재와 빌드된 실행 환경은 다릅니다. 빌드 결과와 현재 Bash에서 install/setup.bash를 읽었는지 확인합니다. node 목록과 실제 메시지 수신도 구분해서 점검합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**ROS 준비 — 노트북의 Python 모의 실험과 실제 ROS 실행 구분:** 아래 Python 셀은 새 fixture와 loader를 직접 만들므로 앞 노트북의 커널 변수·TODO 결과가 필요 없습니다. 실제 node는 **같은 개인 컨테이너의 JupyterLab Terminal** 에서 실행합니다. Ollama는 필요 없습니다.

[01_ros_basics.ipynb의 3-1~3-3절](01_ros_basics.ipynb)에 workspace 빌드와 `source`의 설명이 있으며, 필요한 절차는 다음과 같습니다. 빌드는 `ros_ws/src`의 패키지를 `Data/build/08.ROS/install`에 설치·연결합니다. 첫 사용 또는 install 파일이 없을 때, 패키지 설정·실행 이름이 바뀌었을 때 다음을 실행하고 성공을 확인합니다. 같은 자료 폴더·호환 이미지의 빌드 결과가 남아 있으면 재부팅만을 이유로 다시 빌드할 필요는 없습니다.

```bash
bash
cd /workspace
bash 08.ROS/scripts/build_workspace.sh
```

**새 터미널 A·B·C마다** 아래를 적용합니다. 이 설정은 현재 터미널에만 적용되며 다른 터미널·Python 커널로 전파되지 않습니다. 21은 로컬 예시이고 원격 개인 실습은 22 등 배정값을 씁니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 pkg executables nepes_ros_lab
```

`status_talker`, `image_monitor`, `safe_turtle_controller` 등 실행 이름이 보이면 준비된 것입니다. 기본 ROS는 첫 source, 수업 패키지는 두 번째 source가 찾게 합니다. `Package not found`는 build 성공·현재 install source를 확인합니다. 같은 실습의 모든 터미널은 같은 Domain 번호를 사용합니다. 로컬 GUI를 함께 관찰할 때는 두 수강생이 로컬 컨테이너의 화면을 함께 보고, 원격 개인 Domain에서 로컬 토픽이 보일 것으로 기대하지 않습니다. 이전 실습의 source/controller·카메라를 종료한 뒤 다음 실행을 시작합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[02.Python](../02.Python/README.md)에서 필요한 내용은 함수의 인자→검사→dict 반환, `None`·bool·숫자 타입 구분입니다. Python dict는 필드 이름과 값을 묶고 실제 ROS의 `std_msgs/String`은 그 dict를 JSON 문자열로 담아 보낼 수 있습니다. 아래 `MiniBus`는 한 Python 프로세스의 개념 실험이며 DDS·실제 ROS 네트워크를 검증하지 않습니다. 실제 발행·수신 명령은 이 노트북의 3절에 별도로 있습니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–4절의 수업 핵심 경로** → [3일차 슬라이드 41쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s037) · [배포 PDF 41쪽](../슬라이드/3일차.pdf#page=41)에서 50쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 입력 토픽·처리 결과·출력 명령을 확인하고 종료합니다.
- **단원 마지막 연결 질문:** 입력 노드를 중단하고 마지막 관측 시각, STOP, 복구 후 새 관측을 확인합니다. 운영자가 어느 로그부터 볼지 설명합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

[ROS 터미널 운영표](../08.ROS/실행안내.md)를 사용하세요. 모의 입력은 이동·정지를, 실제 카메라는 인식·승인 또는 거부 이유를 확인합니다. 

08의 MediaPipe confidence=null 이동은 기본 controller에서 거부됩니다. 입력 중단 실험은 실행안내의 source·controller 분리 명령으로 진행합니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 시간 지도

환경 5분 → ROS가 필요한 이유 8분 → Node/Topic/Message 12분 → publish/subscribe 10분 → TODO 15분 → Domain ID·실제 명령 10분.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "08.ROS").is_dir():
            return candidate
    raise RuntimeError("08.ROS를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "08.ROS"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('rclpy', 'cv2', 'cv_bridge')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. 세 용어를 한 문장으로 읽기

```text
/sensor node가 /temperature topic에 Temperature message를 publish한다.
/monitor node는 같은 topic을 subscribe하고 도착할 때 callback을 실행한다.
```

- **Node** 는 계산을 수행하는 실행 참가자입니다.
- **Topic** 은 계속 생기는 데이터를 전달하는 논리 이름입니다.
- **Message** 는 전달할 field와 type의 계약입니다.
- Publisher와 subscriber는 서로의 Python 함수를 직접 호출하지 않습니다. 실제 ROS에서는 다른 프로세스·장치에 있을 수 있고 DDS 계층이 발견과 전달을 맡습니다.

Node로 책임을 나누면 카메라, 인식, 제어 중 어느 단계가 느리거나 끊겼는지 따로 관찰할 수 있습니다. Topic에 subscriber가 없어도 publisher는 보낼 수 있으며 모든 message가 반드시 보존되는지는 QoS에 달렸습니다.


In [ ]:
# 구분: 수업 핵심 셀
from common.mini_ros import MiniBus

bus = MiniBus()
bus.add_node("/sensor")
bus.add_node("/monitor")
received = []
bus.create_subscription("/monitor", "/temperature", received.append)
publish = bus.create_publisher("/sensor", "/temperature")
payload = {"temperature_c": 25}
sent = publish(payload)
payload["temperature_c"] = 999
print("sequence:", sent.sequence, "received:", received[0].payload)
print(json.dumps(bus.graph(), ensure_ascii=False, indent=2))
assert received[0].payload["temperature_c"] == 25


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 2. TODO — Message schema

[exercises/message_contract.py](exercises/message_contract.py)의 `make_temperature_message`를 구현합니다.

| field | 계약 |
|---|---|
| `device_id` | 앞뒤 공백 없는 비어 있지 않은 문자열 |
| `sequence`, `captured_ms` | bool이 아닌 0 이상 정수 |
| `temperature_c` | bool·문자열·NaN·Inf가 아닌 유한수, 원본 보존 |
| `status` | 70.0도 이상 `hot`, 그 아래 `normal` |

`69.999`와 `70.0` 양쪽을 검사합니다. 잘못된 값을 임의 기본값으로 바꾸지 말고 명시적 예외로 거부합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `make_temperature_message(device_id,sequence,temperature_c,captured_ms)`는 `('sensor-1',0,70.0,1000)`에서 `{'device_id':'sensor-1','sequence':0,'temperature_c':70.0,'status':'hot','captured_ms':1000}`을 반환합니다. JSON 문자열이 아닌 dict입니다.
**작성 흐름 힌트:** ID 검사(공백을 자동 수정하지 않고 거부) → 두 정수의 bool 제외·범위 검사 → 온도 숫자·유한성 검사 → >=70.0으로 status 결정 → 새 dict. ROS publish 코드는 이 함수 밖에 있습니다.
**설명할 지점:** “측정 온도와 우리가 붙인 hot이라는 해석을 분리합니다. 예쁜 문자열 출력이 아니라 필드 타입까지 맞는 메시지를 만듭니다.”
**작성 범위·통과 기준:** 함수는 센서·ROS를 실행하지 않고 입력 네 개를 검증해 정확히 다섯 필드의 새 dict를 만듭니다. `math.isfinite`를 사용하려면 starter에 직접 `import math`합니다. 온도는 숫자 값을 보존하며 float로 반환해도 됩니다. 잘못된 타입은 `TypeError` 또는 `ValueError`, NaN·Inf·범위/ID 위반은 `ValueError`로 거부합니다. 바로 아래 학습자 셀은 정상 값·70도 경계·ID/정수/온도의 모든 오류 사례를 호출하고 PASS를 출력해야 합니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
student = load_file("student_ros_message", "exercises/message_contract.py")
below = student.make_temperature_message("oven_1", 0, 69.999, 100)
at = student.make_temperature_message("oven_1", 1, 70.0, 101)
assert below == {"device_id": "oven_1", "sequence": 0, "temperature_c": 69.999, "status": "normal", "captured_ms": 100}
assert at["status"] == "hot" and at["temperature_c"] == 70.0

def expect_message_error(args, label):
    try:
        student.make_temperature_message(*args)
    except (TypeError, ValueError):
        return
    raise AssertionError(f"{label} 입력을 거부해야 합니다: {args!r}")

for bad_device in ("", " oven_1", "oven_1 ", None, 7):
    expect_message_error((bad_device, 0, 25.0, 100), "device_id")
for bad_sequence in (True, -1, 1.5, "1", None):
    expect_message_error(("oven_1", bad_sequence, 25.0, 100), "sequence")
for bad_captured in (True, -1, 1.5, "100", None):
    expect_message_error(("oven_1", 0, 25.0, bad_captured), "captured_ms")
for bad in (True, float("nan"), float("inf"), "70"):
    expect_message_error(("oven_1", 2, bad, 102), "temperature_c")
assert set(below) == {"device_id", "sequence", "temperature_c", "status", "captured_ms"}
print("PASS: 전체 schema, 문자열·정수·유한수, 70도 경계와 원본값")


 > **구분: 수업 핵심**

## 참고 정답

강사가 안내하는 시점에 아래 참고 정답과 해설을 확인합니다. [solutions/message_contract.py](solutions/message_contract.py)에 입력 검사·계산·반환이 직접 구현되어 있으며, 아래 접힌 Markdown에도 같은 코드가 있습니다. 정답 검증과 학습자 TODO 검증은 별도입니다.


In [ ]:
# 구분: 수업 핵심 셀
reference = load_file("reference_ros_message", "solutions/message_contract.py")
assert reference.make_temperature_message("oven_1", 0, 69.999, 100)["status"] == "normal"
assert reference.make_temperature_message("oven_1", 1, 70.0, 101)["status"] == "hot"
print("PASS: 참고 정답 검증")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/message_contract.py](solutions/message_contract.py)의 `make_temperature_message`입니다. 입력 검사와 상태 판정을 이 함수 본문에 직접 구현합니다.

장비명은 앞뒤 공백 없는 문자열, sequence·captured_ms는 bool이 아닌 0 이상 정수인지 확인합니다. 온도는 유한한 숫자로 바꾸고 70.0 이상이면 hot, 아니면 normal입니다. 69.999는 normal, 70.0은 hot입니다. 새 사전에 원래 측정값과 해석 상태·시각을 함께 담습니다.

sequence는 메시지 순번이고 captured_ms는 촬영/측정 시각이므로 바꿔 쓰지 않습니다. Python 모의 버스 통과 뒤 실제 ROS 송수신도 별도로 확인합니다.

**확인 질문:** 온도 값만 보내는 것보다 장비명·시각을 함께 보내면 어떤 오래된 데이터 문제를 찾을 수 있나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/message_contract.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: 장비·순번·온도·시각을 검증한 뒤 새 메시지 dict를 반환합니다.
70.0 이상은 hot이며 원래 측정값과 해석 상태를 함께 보존합니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
import math
from typing import Any

def _number(value: Any, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(f"{name}은 bool이 아닌 숫자여야 합니다")
    value = float(value)
    if not math.isfinite(value):
        raise ValueError(f"{name}은 유한한 숫자여야 합니다")
    return value


def make_temperature_message(
    device_id: str,
    sequence: int,
    temperature_c: float,
    captured_ms: int,
) -> dict[str, Any]:
    """센서 값과 해석을 분리한 고정 schema 메시지를 만든다."""
    if not isinstance(device_id, str) or not device_id or device_id != device_id.strip():
        raise ValueError("device_id는 앞뒤 공백이 없는 비어 있지 않은 문자열이어야 합니다")
    if isinstance(sequence, bool) or not isinstance(sequence, int) or sequence < 0:
        raise ValueError("sequence는 0 이상의 정수여야 합니다")
    if isinstance(captured_ms, bool) or not isinstance(captured_ms, int) or captured_ms < 0:
        raise ValueError("captured_ms는 0 이상의 정수여야 합니다")
    temperature = _number(temperature_c, "temperature_c")
    return {
        "device_id": device_id,
        "sequence": sequence,
        "temperature_c": temperature,
        "status": "hot" if temperature >= 70.0 else "normal",
        "captured_ms": captured_ms,
    }
```

</details>
<!-- todo-review:answer:end -->


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 3. 실제 ROS 경로 — 빌드부터 실행까지

<!-- procedure:lesson-26-436547:start -->
![빌드부터 실제 메시지 확인까지](../교재그림/절차/lesson-26.svg)

각 터미널에서 같은 개인 Domain을 씁니다. topic 이름이 보여도 메시지가 도착하는지 별도로 확인합니다.
<!-- procedure:lesson-26-436547:end -->


앞의 MiniBus는 Python 안에서 통신을 모의한 것입니다. 이제 별도 프로세스인 실제 ROS node를 실행합니다. **아래 명령은 개인 실습 컨테이너의 JupyterLab Terminal에서 실행** 합니다. Python 코드 셀이나 Jetson 호스트 터미널에 그대로 입력하지 않습니다. A·B·C는 같은 개인 컨테이너에서 연 서로 다른 터미널입니다.

### 3-1. Workspace 빌드는 무엇을 하나요?

**Workspace(작업 공간)** 는 ROS 패키지의 소스와 빌드 결과를 관리하는 공간입니다. **Package(패키지)** 는 node 코드·의존성·실행 설정을 묶은 단위이며, 여기서는 `nepes_ros_lab`입니다.

빌드는 소스를 읽고 필요한 처리와 설치를 수행하여 ROS가 패키지와 실행 파일을 찾을 수 있게 준비하는 작업입니다. C++ 패키지는 컴파일·링크를 거칩니다. 이번 Python 패키지는 Python 코드를 설치·연결하고 `status_talker` 같은 실행 이름을 해당 함수에 연결합니다. 빌드만으로 node가 실행되지는 않습니다.

```text
08.ROS/ros_ws/src/nepes_ros_lab의 소스·설정
  → colcon build (빌드 도구)
  → Data/build/08.ROS/install의 패키지·실행 진입점
  → install/setup.bash를 현재 터미널에서 읽기
  → ros2 run nepes_ros_lab status_talker
```

`ros2 run` 뒤의 첫 이름은 패키지, 두 번째 이름은 실행 이름입니다. [setup.py](ros_ws/src/nepes_ros_lab/setup.py)의 `console_scripts`에 있는 `status_talker = nepes_ros_lab.status_talker:main`이 실행 이름을 Python의 `main()` 함수와 연결합니다.

| 경로 (`/workspace` 기준) | 역할 |
|---|---|
| `08.ROS/ros_ws/src` | 원본 패키지 소스·설정 |
| `Data/build/08.ROS/build` | 빌드 중간 결과 |
| `Data/build/08.ROS/install` | ROS가 사용할 패키지·실행 진입점·환경 설정 |
| `Data/build/08.ROS/log` | 빌드 진행·오류 로그 |

### 3-2. 처음 한 번 — 터미널 A에서 빌드

JupyterLab에서 Terminal을 엽니다. 프롬프트의 `$`는 입력하지 않습니다. 현재 셸이 `sh`일 수도 있으므로 **먼저 아래 한 줄을 실행** 해 Bash로 들어갑니다.

```bash
bash
```

이어서 같은 터미널 A에서 실행합니다.

```bash
bash
cd /workspace
bash 08.ROS/scripts/build_workspace.sh
```

[빌드 스크립트](scripts/build_workspace.sh)는 ROS 환경을 준비한 뒤 `colcon build --symlink-install`을 실행합니다. 성공 요약과 `빌드 완료` 안내를 확인하고 다음 단계로 갑니다. 실패했다면 뒤의 실행 명령으로 넘어가지 말고 첫 오류와 로그를 확인합니다.

빌드 산출물은 호스트에 연결된 `Data`에 남으므로, 같은 자료 폴더와 호환되는 이미지라면 재부팅했다고 매번 빌드할 필요는 없습니다. 패키지 설정·의존성·실행 이름을 바꾸거나 산출물이 없으면 다시 빌드합니다. `--symlink-install`은 개발 중 Python 소스 변경을 반영하기 쉽게 연결하지만, 실행 중인 node는 변경 후 다시 시작해야 합니다.

### 3-3. 새 터미널마다 — ROS 환경과 개인 Domain 적용

터미널 B·C도 열고 **A·B·C 모두에서 아래 코드 블록 전체를 실행** 합니다. 첫 줄의 `bash`가 Bash 셸을 엽니다. 21은 로컬 예시이며 원격 사용자는 22 등 본인 배정값으로 바꿉니다. 같은 개인 실습의 A·B·C는 같은 번호를 사용합니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21
```

| 설정 | 현재 터미널에서 하는 일 |
|---|---|
| `/opt/ros/humble/setup.bash` | 기본 ROS 설치 경로를 반영하여 `ros2` 명령과 기본 패키지를 찾게 함 |
| `Data/build/08.ROS/install/setup.bash` | 방금 빌드한 `nepes_ros_lab` 패키지를 찾게 함 |
| `ROS_DOMAIN_ID` | 같은 번호의 ROS 참가자끼리 발견·통신하도록 구분 |

`source`는 스크립트를 **현재 셸에서 읽어 환경 변수를 적용** 합니다. 빌드 스크립트 내부의 `source`는 별도 Bash 프로세스에만 적용되므로, 빌드가 끝난 뒤 현재 터미널에서도 위 설정을 해야 합니다. A에서 설정해도 B·C나 이미 열린 Jupyter Python 커널에 자동 전달되지 않습니다.

준비 상태를 각 터미널에서 확인합니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 원격은 22 등 본인 배정값
command -v ros2
ros2 pkg executables nepes_ros_lab
echo "$ROS_DOMAIN_ID"
```

첫 명령은 `ros2`의 경로, 두 번째는 `nepes_ros_lab status_talker`·`status_listener` 등의 목록, 세 번째는 본인 Domain 번호를 출력해야 합니다.

### 3-4. A는 발행, B는 수신, C는 관찰

**터미널 A:** 발행 node를 실행한 상태로 둡니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 원격은 22 등 본인 배정값
ros2 run nepes_ros_lab status_talker
```

**터미널 B:** 수신 node를 실행한 상태로 둡니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 원격은 22 등 본인 배정값
ros2 run nepes_ros_lab status_listener
```

**터미널 C:** 실행 중인 node와 topic을 확인하고 메시지 한 건을 저장합니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 원격은 22 등 본인 배정값
mkdir -p /workspace/Data/outputs/08.ROS
ros2 node list
ros2 topic info /factory/status --verbose
ros2 topic echo /factory/status --once | tee /workspace/Data/outputs/08.ROS/01-factory-status.txt
```

A에서 발행 순번이 증가하고 B에서 수신 로그가 나오는지 확인합니다. C의 마지막 명령은 화면 출력과 같은 내용을 Data 파일에 남깁니다. 실습 종료 시 A·B에서 각각 Ctrl+C를 눌러 node를 종료합니다.

### 오류 메시지로 빠진 단계를 찾기

| 증상 | 확인할 단계 |
|---|---|
| `ros2: not found` | 컨테이너 Terminal인지 확인 → Bash 진입 → 기본 ROS `source` |
| `source: not found` | 현재 셸이 `sh`인지 확인하고 먼저 `bash` 실행 |
| `Package 'nepes_ros_lab' not found` | 빌드 성공 여부와 현재 터미널의 install `source` |
| `No executable found` | 실행 이름 오타와 `setup.py` 등록 확인, 설정 변경 후 재빌드 |
| 설치용 `setup.bash`가 없음 | 빌드 미완료 또는 다른 자료 폴더 사용 여부 확인 |
| 발행되지만 수신되지 않음 | A·B·C의 Domain 번호, topic 이름·타입, 프로세스 실행 상태 확인 |


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 4. 완료 점검과 흔한 오류

Node/Topic/Message를 자신의 예로 설명하고 publisher가 subscriber 존재를 미리 알 필요가 없는 이유를 말합니다. `ros2: command not found`는 base setup, package not found는 build/install setup, topic 미발견은 Domain ID·network·name·type·QoS 순으로 확인합니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** message 검증·build·A/B/C 발행/수신/관찰 확인

[3일차 슬라이드 41쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s037) · [배포 PDF 41쪽](../슬라이드/3일차.pdf#page=41)로 돌아가 **41–50쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

## 자습 — Topic, Service, Action을 요구사항으로 선택하기 / 약 35분

이 자습은 통신 이름을 외우는 대신 데이터가 **반복되는지** , 요청에 **한 번의 답** 이 필요한지, 작업 도중 **진행률과 취소** 가 필요한지를 보고 방식을 선택하는 연습입니다.

| 방식 | 보내는 모양 | 상대의 응답 | 대표 예 |
|---|---|---|---|
| Topic | 시간이 지나며 계속 publish | 개별 응답을 기다리지 않음 | camera frame, 온도, 현재 pose |
| Service | request 한 번 | response 한 번 | 설정 읽기, 짧은 reset |
| Action | goal 한 번 | feedback 여러 번, result 한 번, cancel 가능 | 목표 위치로 이동 |

진행 순서는 다음과 같습니다.

1. 아래 fixture의 세 상황을 읽고 아직 실행하지 않은 상태에서 답을 종이에 적습니다.
2. TODO 셀의 None을 topic, service, action 중 하나로 바꿉니다.
3. 검증 셀을 실행해 선택과 이유를 확인합니다.
4. 예상 결과와 비교한 뒤 기록 셀을 실행합니다.

Action도 안전 기능 자체는 아닙니다. Goal 검사, timeout, cancel 뒤 STOP, 속도 제한은 별도 제어 계약으로 남습니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
STUDY_COMMUNICATION_CASES = [
    {
        "id": "camera_frames",
        "question": "30 fps 영상처럼 새 값이 계속 생기며 각 frame마다 답을 기다리지 않는다.",
    },
    {
        "id": "read_mode",
        "question": "현재 운전 mode를 한 번 물어보고 짧은 응답 한 번을 받는다.",
    },
    {
        "id": "move_goal",
        "question": "목표점까지 수 초간 이동하며 진행률, 결과, 중간 취소가 필요하다.",
    },
]
EXPECTED_CHANNELS = {
    "camera_frames": "topic",
    "read_mode": "service",
    "move_goal": "action",
}
for case in STUDY_COMMUNICATION_CASES:
    print(f"{case['id']}: {case['question']}")


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### TODO — 세 상황의 통신 방식 선택

다음 셀에서 각 None을 topic, service, action 중 하나로 바꾸세요. 먼저 다음 질문을 순서대로 적용합니다.

1. 데이터가 계속 생기는가?
2. 요청 하나에 응답 하나면 충분한가?
3. 오래 걸리는 목표이며 feedback이나 cancel이 필요한가?

대문자나 한국어를 넣지 않습니다. 검증은 정확한 소문자 문자열을 사용합니다.

**입력과 편집 위치:** 바로 위 `STUDY_COMMUNICATION_CASES`의 ID 세 개는 ① camera_frames=30 FPS 반복 입력·응답 대기 없음, ② read_mode=짧은 요청 한 번·응답 한 번, ③ move_goal=시간이 걸리는 목표·진행률·취소 필요입니다. 아래 `MY_CHANNEL_CHOICES`의 key는 보존하고 값 세 개만 소문자 문자열로 바꿉니다. 별도 함수 구현은 필요 없습니다. 수정 셀→검증 셀 순으로 실행해 None이 없고 세 상황이 모두 PASS인지 확인한 뒤 마지막 저장 셀로 `01-topic-service-action.json`의 completed=True를 확인합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
MY_CHANNEL_CHOICES = {
    "camera_frames": None,  # TODO: "topic", "service", "action" 중 선택
    "read_mode": None,      # TODO
    "move_goal": None,      # TODO
}
print("현재 답:", MY_CHANNEL_CHOICES)


In [ ]:
# 구분: 수업 후 복습·자습 셀
missing = [name for name, choice in MY_CHANNEL_CHOICES.items() if choice is None]
study_passed = False
if missing:
    print("TODO가 남았습니다:", missing)
    print("None을 모두 바꾼 뒤 이 셀을 다시 실행하세요.")
else:
    assert set(MY_CHANNEL_CHOICES) == set(EXPECTED_CHANNELS), "상황 ID를 바꾸지 마세요."
    for name, expected in EXPECTED_CHANNELS.items():
        actual = MY_CHANNEL_CHOICES[name]
        assert actual in {"topic", "service", "action"}, f"{name}: 허용하지 않는 답 {actual!r}"
        assert actual == expected, f"{name}: {actual!r} 대신 요구사항을 다시 읽어 보세요."
    study_passed = True
    print("PASS: 반복 stream, 짧은 요청/응답, 장시간 goal을 구분했습니다.")


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 예상 결과와 해설

- camera_frames → topic: frame이 계속 생기고 publisher가 매 frame의 개별 답을 기다리지 않습니다.
- read_mode → service: 짧은 질문 하나와 응답 하나가 한 쌍입니다.
- move_goal → action: 오래 걸리며 feedback, result, cancel이 필요합니다.

검증 성공 시 PASS 문장이 한 줄 출력됩니다. 틀렸다면 오류에 표시된 상황만 다시 읽고 수정하세요.


In [ ]:
# 구분: 수업 후 복습·자습 셀
study_output = ROOT / "Data/outputs/08.ROS/01-topic-service-action.json"
study_output.parent.mkdir(parents=True, exist_ok=True)
study_record = {
    "exercise": "topic_service_action",
    "choices": MY_CHANNEL_CHOICES,
    "expected": EXPECTED_CHANNELS,
    "completed": MY_CHANNEL_CHOICES == EXPECTED_CHANNELS,
}
study_output.write_text(
    json.dumps(study_record, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
assert study_output.is_file()
print("saved:", study_output)
print("completed:", study_record["completed"])
